# Commercial IP Downstream Evaluation — Formal Training Period

**Evaluation window:**
- In-time (train / val / test): `2024-11-20` → `2025-06-30` (~7 months)
- Out-of-time (OOT): `2025-07-01` → `2025-09-30`

**Data sources:**
| Role | BigQuery Table |
|---|---|
| New TE Embeddings | `edp-prod-storage.edp_ent_sdoheir_cns.a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930` |
| Production RAP Embeddings | `edp-prod-storage.edp_ent_sdoheir_cns.enhanced_rap_cp_emb_history_wide_4_te_fromal_eval` |
| Tabular features + outcome | `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Commercial_final_dataset_4_te_formal_evaluation_20241120_20250930` |

**Split logic (same as baseline pipeline):**
- Train: `ind_id_last_digit` 0–7 AND `index_dt` ≤ cutoff
- Val: `ind_id_last_digit` 8 AND `index_dt` ≤ cutoff
- Test: `ind_id_last_digit` 9 AND `index_dt` ≤ cutoff
- OOT: `index_dt` > cutoff (all digits)
- OOT-strict: `index_dt` > cutoff AND `ind_id_last_digit` = 9

**Feature sets evaluated:** `embedding_only` | `tabular_only` | `hybrid`

**Comparisons:**
1. New TE embeddings vs Production RAP embeddings (embedding-only)
2. Which embeddings produce better hybrid effect when combined with tabular features
3. SHAP feature importance — which embedding dimensions matter most for new TE

## 1. Imports

In [ ]:
import os
import sys
import glob
import time
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

import google.auth
from google.cloud import bigquery

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
)
from sklearn.base import clone
from catboost import CatBoostClassifier, Pool
import shap

warnings.filterwarnings('ignore')

credentials, project = google.auth.default()
client = bigquery.Client()
print(f'credentials: {credentials}')
print(f'project: {project}')

## 2. Constants

In [ ]:
# =============================================================================
# DATA SOURCES
# =============================================================================
PROJECT_ID   = "edp-prod-storage"
DATASET_ID   = "edp_ent_sdoheir_cns"

# ==================Convert production embeddings to wide format=========
# DECLARE sql STRING;

# SET sql = (
#   SELECT '''
# CREATE OR REPLACE TABLE `edp-prod-storage.edp_ent_sdoheir_cns.enhanced_rap_cp_emb_history_wide_4_te_fromal_eval` AS
# SELECT
#   individual_id,
#   index_dt,
#   ''' || STRING_AGG(
#         FORMAT('SAFE_CAST(embs[SAFE_OFFSET(%d)] AS FLOAT64) AS embedding_%d', idx, idx),
#         ',\n  '
#       ) || '''
# FROM `anbc-hcb-prod.clin_analytics_hcb_prod.enhanced_rap_cp_emb_history`
# '''
#   FROM UNNEST(GENERATE_ARRAY(0, 255)) AS idx
# );
# EXECUTE IMMEDIATE sql;

EMBEDDING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".a964286_exp_round10_exp2b_commercial_embeddings_20241120_20250930"
)
PROD_EMBEDDING_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".enhanced_rap_cp_emb_history_wide_4_te_fromal_eval"
)
FEATURES_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}"
    ".a834793_Commercial_final_dataset_4_te_formal_evaluation_20241120_20250930"
)

# =============================================================================
# SPLIT CONFIGURATION
# In-time:  2024-11-20 --> 2025-06-30
# OOT:      2025-07-01 --> 2025-09-30
# =============================================================================
OOT_CUTOFF_DATE = "2025-06-30"   # inclusive upper bound for in-time splits

# =============================================================================
# TARGET & SPLIT KEY
# =============================================================================
TARGET_COLUMN          = "ip6"
NEGATIVE_DOWNSAMPLE_RATIO = 10   # match baseline 10:1 negative sampling

# =============================================================================
# COLUMNS TO EXCLUDE FROM FEATURES
# (identical to baseline pipeline — prevents leakage)
# =============================================================================
EXCLUDE_COLUMNS = frozenset([
    # Keys and identifiers
    'individual_id', 'member_id', 'index_dt', 'birth_dt', 'feature_end_dt',

    # Outcome columns
    'ip6', 'sum_ip6_admits', 'sum_ip6_los', 'sum_ip6_acu_days',

    # Eligibility / continuity flags
    'mon_3_include', 'mon_6_include', 'mon_12_include',
    'exclude_ip', 'include_post_6_status',

    # Split key
    'ind_id_last_digit',

    # Leakage columns (cost amounts, outreach flags)
    'clm_allowed_amt_1yr', 'clm_allowed_amt_2yr', 'clm_allowed_amt_3mo', 'clm_allowed_amt_6mo',
    'clm_paid_amt_1yr', 'clm_paid_amt_2yr', 'clm_paid_amt_3mo', 'clm_paid_amt_6mo',
    'clm_par_allowed_amt_1yr', 'clm_par_allowed_amt_2yr', 'clm_par_allowed_amt_3mo', 'clm_par_allowed_amt_6mo',
    'clm_par_paid_amt_1yr', 'clm_par_paid_amt_2yr', 'clm_par_paid_amt_3mo', 'clm_par_paid_amt_6mo',
    'clm_srv_copay_amt_1yr', 'clm_srv_copay_amt_3mo', 'clm_srv_copay_amt_6mo',
    'covid_19', 'hpd_major_flag', 'chronic',
    'txt_member', 'txt_referral', 'txt_1yr_outreach', 'talked',
])

## 3. Metric Functions

In [ ]:
# =============================================================================
# METRIC FUNCTIONS
# =============================================================================

def lift_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Lift = precision@k / baseline_prevalence."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    precision_at_k = y_true[top_k_indices].mean()
    baseline = y_true.mean()
    return precision_at_k / baseline if baseline > 0 else 0.0


def true_positives_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> int:
    """Count true positives in top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return int(y_true[top_k_indices].sum())


def precision_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Precision (PPV) in top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return float(y_true[top_k_indices].mean())


def compute_split_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """Compute all metrics for a single split."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    return {
        'auc_roc':       roc_auc_score(y_true, y_prob),
        'auc_pr':        average_precision_score(y_true, y_prob),
        'brier':         brier_score_loss(y_true, y_prob),
        'lift_1pct':     lift_at_percentage(y_true, y_prob, 0.01),
        'lift_5pct':     lift_at_percentage(y_true, y_prob, 0.05),
        'lift_10pct':    lift_at_percentage(y_true, y_prob, 0.10),
        'tp_1pct':       true_positives_at_percentage(y_true, y_prob, 0.01),
        'precision_1pct': precision_at_percentage(y_true, y_prob, 0.01),
        'n_samples':     len(y_true),
        'n_positives':   int(y_true.sum()),
        'prevalence':    float(y_true.mean()),
    }

## 4. Data Preparation Functions

In [ ]:
# =============================================================================
# DATA PREPARATION FUNCTIONS
# =============================================================================

def load_embeddings_from_bigquery(
    table_id: str,
    project_id: Optional[str] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Load embeddings from a BigQuery table.
    Returns DataFrame with individual_id, index_dt, embedding_0...embedding_N.
    """
    bq_client = bigquery.Client(project=project_id) if project_id else bigquery.Client()
    query = f"SELECT * FROM `{table_id}`"
    if verbose:
        print(f"  Loading embeddings from BigQuery: {table_id}")
    df = bq_client.query(query).to_dataframe()

    embedding_cols = [c for c in df.columns if c.startswith('emb')]
    if embedding_cols and '_' in embedding_cols[0]:
        embedding_cols = sorted(embedding_cols, key=lambda c: int(c.split('_')[1]))
    else:
        embedding_cols = sorted(embedding_cols, key=lambda c: int(c.replace('emb', '')))
        rename_map = {c: f"embedding_{c.replace('emb', '')}" for c in embedding_cols}
        df.rename(columns=rename_map, inplace=True)
        embedding_cols = [rename_map[c] for c in embedding_cols]

    if verbose:
        print(f"  Loaded {len(df):,} rows with {len(embedding_cols)} embedding dims")

    df['index_dt'] = pd.to_datetime(df['index_dt']).dt.strftime('%Y-%m-%d')
    return df[['individual_id', 'index_dt'] + embedding_cols]


def join_embeddings_with_features(
    emb_df: pd.DataFrame,
    df_features: pd.DataFrame,
) -> pd.DataFrame:
    """
    Inner-join embeddings with features on (individual_id, index_dt).
    Deduplicates on (individual_id, index_dt), keeping last occurrence.
    """
    df_features = df_features.copy()
    df_features['index_dt'] = pd.to_datetime(df_features['index_dt']).dt.strftime('%Y-%m-%d')
    emb_df = emb_df.copy()
    emb_df['index_dt'] = pd.to_datetime(emb_df['index_dt']).dt.strftime('%Y-%m-%d')
    emb_df['individual_id'] = emb_df['individual_id'].astype(str)
    df_features['individual_id'] = df_features['individual_id'].astype(str)

    df_merged = df_features.merge(emb_df, on=['individual_id', 'index_dt'], how='inner')
    df_merged = df_merged.drop_duplicates(subset=['individual_id', 'index_dt'], keep='last')
    return df_merged


def create_data_splits(
    df: pd.DataFrame,
    oot_cutoff_date: str = OOT_CUTOFF_DATE,
) -> Dict[str, pd.DataFrame]:
    """
    Create train / val / test / OOT splits.

    Split logic (matches baseline pipeline):
        - Train:      ind_id_last_digit 0-7  AND  index_dt <= cutoff
        - Val:        ind_id_last_digit 8    AND  index_dt <= cutoff
        - Test:       ind_id_last_digit 9    AND  index_dt <= cutoff
        - OOT:        index_dt > cutoff  (all digits)
        - OOT-strict: index_dt > cutoff  AND  ind_id_last_digit 9
    """
    df = df.copy()
    df['_index_dt_parsed'] = pd.to_datetime(df['index_dt'])
    oot_cutoff = pd.to_datetime(oot_cutoff_date)

    splits = {
        'train': df[
            df['ind_id_last_digit'].isin([0, 1, 2, 3, 4, 5, 6, 7]) &
            (df['_index_dt_parsed'] <= oot_cutoff)
        ],
        'val': df[
            (df['ind_id_last_digit'] == 8) &
            (df['_index_dt_parsed'] <= oot_cutoff)
        ],
        'test': df[
            (df['ind_id_last_digit'] == 9) &
            (df['_index_dt_parsed'] <= oot_cutoff)
        ],
        'oot': df[df['_index_dt_parsed'] > oot_cutoff],
        'oot_strict': df[
            (df['_index_dt_parsed'] > oot_cutoff) &
            (df['ind_id_last_digit'] == 9)
        ],
    }

    print("Data splits created:")
    for name, split_df in splits.items():
        if len(split_df) > 0:
            prevalence = split_df[TARGET_COLUMN].mean() * 100
            print(f"  {name}: {len(split_df):,} rows, "
                  f"{int(split_df[TARGET_COLUMN].sum()):,} positives ({prevalence:.2f}%)")
        else:
            print(f"  {name}: EMPTY")

    for key in splits:
        splits[key] = splits[key].drop(columns=['_index_dt_parsed'])
    return splits


def identify_feature_columns(
    df: pd.DataFrame,
) -> Tuple[List[str], List[str]]:
    """
    Returns (embedding_features, tabular_features).
    Embedding features are columns starting with 'embedding_'.
    Tabular features are everything else after applying EXCLUDE_COLUMNS.
    """
    all_cols = set(df.columns)
    embedding_features = sorted([c for c in all_cols if c.startswith('embedding_')])
    excluded = EXCLUDE_COLUMNS | set(embedding_features) | {'_exp_name', 'index_dt_parsed', '_index_dt_parsed'}
    tabular_features = sorted([
        c for c in all_cols
        if c not in excluded and c != TARGET_COLUMN
    ])
    return embedding_features, tabular_features


def downsample_negatives(
    X: pd.DataFrame,
    y: pd.Series,
    ratio: int = NEGATIVE_DOWNSAMPLE_RATIO,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Downsample the negative class to `ratio` negatives per positive.
    Matches baseline pipeline which used a pre-sampled table at 10:1.
    """
    np.random.seed(random_state)
    pos_indices = X.index[y == 1].tolist()
    neg_indices = X.index[y == 0].tolist()
    n_positives = len(pos_indices)
    n_negatives = len(neg_indices)
    target_n_negatives = int(n_positives * ratio)

    if n_negatives <= target_n_negatives:
        print(f"  Downsampling: no action needed (current ratio {n_negatives/n_positives:.1f}:1)")
        return X, y

    sampled_neg = np.random.choice(neg_indices, size=target_n_negatives, replace=False)
    keep = pos_indices + sampled_neg.tolist()
    X_res = X.loc[keep].copy()
    y_res = y.loc[keep].copy()
    shuffle_idx = np.random.permutation(len(X_res))
    X_res = X_res.iloc[shuffle_idx].reset_index(drop=True)
    y_res = y_res.iloc[shuffle_idx].reset_index(drop=True)
    print(f"  Downsampling: {n_negatives}:{n_positives} ({n_negatives/n_positives:.1f}:1) "
          f"→ {target_n_negatives}:{n_positives} ({ratio}:1)")
    return X_res, y_res


def prepare_features(
    df: pd.DataFrame,
    feature_cols: List[str],
) -> Tuple[pd.DataFrame, pd.Series]:
    """Build (X, y) for a given split, filling missing values."""
    X = df[feature_cols].copy()
    y = df[TARGET_COLUMN].astype(int)
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    X[numeric_cols] = X[numeric_cols].fillna(0)
    cat_cols = X.select_dtypes(include=['object', 'category']).columns
    X[cat_cols] = X[cat_cols].fillna('missing')
    return X, y

## 5. PreparedData Container & prepare_evaluation_data

In [ ]:
@dataclass
class PreparedData:
    """Container for prepared evaluation data. Prepare once, evaluate many models."""
    X_splits: Dict[str, pd.DataFrame]
    y_splits: Dict[str, pd.Series]
    feature_cols: List[str]
    embedding_features: List[str]
    tabular_features: List[str]
    cat_feature_indices: List[int]   # Column indices for CatBoost
    feature_set: str
    embedding_path: str
    downsampled: bool = True


def prepare_evaluation_data(
    df_features: pd.DataFrame,
    embedding_location_path: str = "",
    feature_set: str = 'embedding_only',
    oot_cutoff_date: str = OOT_CUTOFF_DATE,
    downsample_ratio: Optional[float] = None,
    random_state: int = 42,
) -> PreparedData:
    """
    Prepare data once for multiple model evaluations.

    Args:
        df_features: Tabular features + outcome DataFrame from BigQuery.
        embedding_location_path: Full BigQuery table ID for embeddings
                                 (auto-detected by '.' count).
                                 Pass '' for tabular_only with full dataset.
        feature_set: 'embedding_only' | 'tabular_only' | 'hybrid'
        oot_cutoff_date: Upper bound for in-time splits.
        downsample_ratio: Negative-to-positive ratio for training set rebalancing.
                          Use 10.0 to match baseline.
        random_state: Seed for downsampling.

    Returns:
        PreparedData with X_splits, y_splits, and column metadata.
    """
    valid_feature_sets = {'embedding_only', 'tabular_only', 'hybrid'}
    if feature_set not in valid_feature_sets:
        raise ValueError(f"feature_set must be one of {valid_feature_sets}")

    total_start = time.time()

    # ------------------------------------------------------------------
    # Step 1: Load embeddings and merge with features
    # ------------------------------------------------------------------
    print(f"\n[Step 1] Loading and merging data (feature_set={feature_set})...")
    step_start = time.time()

    if feature_set != 'tabular_only':
        is_bigquery = embedding_location_path.count('.') >= 2
        print(f"  Loading embeddings from {'BigQuery' if is_bigquery else 'local'}: {embedding_location_path}")
        emb_df = load_embeddings_from_bigquery(embedding_location_path)

        print("  Joining embeddings with features...")
        if feature_set == 'embedding_only':
            df_merged = join_embeddings_with_features(
                emb_df,
                df_features[['individual_id', 'index_dt', TARGET_COLUMN, 'ind_id_last_digit']],
            )
        else:  # hybrid
            df_merged = join_embeddings_with_features(emb_df, df_features)

    else:  # tabular_only
        if embedding_location_path:
            # Scope tabular data to members that have embeddings
            is_bigquery = embedding_location_path.count('.') >= 2
            if is_bigquery:
                print("  Scoping tabular-only to members in embedding table...")
                emb_df = load_embeddings_from_bigquery(embedding_location_path)
                df_merged = join_embeddings_with_features(
                    emb_df[['individual_id', 'index_dt']], df_features
                )
            else:
                raise ValueError("tabular_only with local embedding path not supported here.")
        else:
            # Use the full tabular dataset directly
            df_merged = df_features.copy()
            df_merged['index_dt'] = pd.to_datetime(df_merged['index_dt']).dt.strftime('%Y-%m-%d')
            if 'mon_6_include' in df_merged.columns:
                df_merged = df_merged[df_merged['mon_6_include'] == 1]
            if 'exclude_ip' in df_merged.columns:
                df_merged = df_merged[(df_merged['exclude_ip'] == 0) | (df_merged['exclude_ip'].isna())]
            if 'include_post_6_status' in df_merged.columns:
                df_merged = df_merged[df_merged['include_post_6_status'] == 1]
            df_merged = df_merged.drop_duplicates(subset=['individual_id', 'index_dt'], keep='last')

    print(f"  Merged shape: {df_merged.shape}  ({time.time()-step_start:.1f}s)")

    # ------------------------------------------------------------------
    # Step 2: Create splits
    # ------------------------------------------------------------------
    print(f"\n[Step 2] Creating data splits (OOT cutoff: {oot_cutoff_date})...")
    splits = create_data_splits(df_merged, oot_cutoff_date)

    # ------------------------------------------------------------------
    # Step 3: Identify feature columns
    # ------------------------------------------------------------------
    embedding_features, tabular_features = identify_feature_columns(df_merged)
    if feature_set == 'embedding_only':
        feature_cols = embedding_features
    elif feature_set == 'tabular_only':
        feature_cols = tabular_features
    else:  # hybrid
        feature_cols = tabular_features + embedding_features

    print(f"  {len(feature_cols)} features selected "
          f"({len(embedding_features)} embedding, {len(tabular_features)} tabular)")

    # ------------------------------------------------------------------
    # Step 4: Build (X, y) for each split
    # ------------------------------------------------------------------
    print("\n[Step 4] Preparing feature matrices...")
    X_splits, y_splits = {}, {}
    for split_name, split_df in splits.items():
        if len(split_df) > 0:
            X_splits[split_name], y_splits[split_name] = prepare_features(split_df, feature_cols)

    # ------------------------------------------------------------------
    # Step 5: Downsample negatives in training set
    # ------------------------------------------------------------------
    downsampled = False
    if downsample_ratio is not None and 'train' in X_splits:
        print(f"\n[Step 5] Downsampling training set to {downsample_ratio}:1 ratio...")
        X_splits['train'], y_splits['train'] = downsample_negatives(
            X_splits['train'], y_splits['train'],
            ratio=downsample_ratio, random_state=random_state,
        )
        downsampled = True

    # Pre-compute categorical column indices for CatBoost
    cat_feature_indices = []
    if feature_set != 'embedding_only' and 'train' in X_splits:
        cat_cols = X_splits['train'].select_dtypes(include=['object', 'category']).columns
        cat_feature_indices = [X_splits['train'].columns.get_loc(c) for c in cat_cols]

    print(f"\nData preparation complete ({time.time()-total_start:.1f}s total)")
    return PreparedData(
        X_splits=X_splits,
        y_splits=y_splits,
        feature_cols=feature_cols,
        embedding_features=embedding_features,
        tabular_features=tabular_features,
        cat_feature_indices=cat_feature_indices,
        feature_set=feature_set,
        embedding_path=embedding_location_path,
        downsampled=downsampled,
    )

## 6. Model Evaluation Functions

In [ ]:
# =============================================================================
# MODEL EVALUATION FUNCTIONS
# =============================================================================

def evaluate_model_on_splits(
    model,
    X_splits: Dict[str, pd.DataFrame],
    y_splits: Dict[str, pd.Series],
    apply_scaling: bool = False,
    cat_feature_indices: Optional[List[int]] = None,
) -> Dict[str, Dict[str, float]]:
    """
    Train `model` on the train split, evaluate on val / test / oot / oot_strict.
    Returns a dict keyed by split name → metrics dict.
    """
    model = clone(model)
    X_train, y_train = X_splits['train'], y_splits['train']

    scaler = None
    if apply_scaling:
        scaler = StandardScaler()
        X_train_proc = scaler.fit_transform(X_train)
    else:
        X_train_proc = X_train

    model_type = type(model).__name__
    t0 = time.time()

    if model_type == 'CatBoostClassifier':
        cat_idx = cat_feature_indices or []
        train_pool = Pool(X_train, y_train, cat_features=cat_idx)
        val_pool = Pool(X_splits['val'], y_splits['val'], cat_features=cat_idx)
        model.fit(train_pool, eval_set=val_pool, verbose=0)
    else:
        model.fit(X_train_proc, y_train)

    print(f"  Fit done ({model_type}): {time.time()-t0:.1f}s")

    results = {}
    for split_name in tqdm(['val', 'test', 'oot', 'oot_strict'], desc='Evaluating splits'):
        X_split = X_splits.get(split_name)
        y_split = y_splits.get(split_name)
        if X_split is None or len(X_split) == 0:
            continue
        if apply_scaling and scaler is not None:
            X_proc = scaler.transform(X_split)
        else:
            X_proc = X_split
        if model_type == 'CatBoostClassifier' and cat_feature_indices:
            pool = Pool(X_split, cat_features=cat_feature_indices)
            y_prob = model.predict_proba(pool)[:, 1]
        else:
            y_prob = model.predict_proba(X_proc)[:, 1]
        results[split_name] = compute_split_metrics(np.array(y_split), y_prob)
    return results


def evaluate_with_prepared_data(
    prepared_data: PreparedData,
    ml_model_object: Any,
    exp_name: str,
    apply_scaling: bool = False,
) -> Dict[str, Any]:
    """Evaluate one model using pre-prepared data. Returns flat metrics dict."""
    use_cat_features = (
        prepared_data.feature_set != 'embedding_only' and
        len(prepared_data.cat_feature_indices) > 0
    )
    split_results = evaluate_model_on_splits(
        model=ml_model_object,
        X_splits=prepared_data.X_splits,
        y_splits=prepared_data.y_splits,
        apply_scaling=apply_scaling,
        cat_feature_indices=prepared_data.cat_feature_indices if use_cat_features else None,
    )
    output = {
        'exp_name':   exp_name,
        'model_type': type(ml_model_object).__name__,
        'feature_set': prepared_data.feature_set,
        'n_features':  len(prepared_data.feature_cols),
    }
    for split_name, metrics in split_results.items():
        for metric_name, value in metrics.items():
            output[f'{split_name}_{metric_name}'] = value
    return output


def evaluate_all_experiments(
    experiment_configs: List[Dict],
    df_features: pd.DataFrame,
    downsample_ratio: Optional[float] = None,
) -> pd.DataFrame:
    """
    Evaluate multiple experiments efficiently.
    Data is prepared once per unique (embedding_path, feature_set, downsample_ratio).

    Each config dict must contain:
        embedding_location_path: str
        ml_model_object:         sklearn-compatible model
        exp_name:                str
        feature_set:             str  (default 'embedding_only')
        apply_scaling:           bool (default False)
        downsample_ratio:        float (optional, overrides global)
    """
    groups = defaultdict(list)
    for cfg in experiment_configs:
        key = (
            cfg.get('embedding_location_path', ''),
            cfg.get('feature_set', 'embedding_only'),
            cfg.get('downsample_ratio', downsample_ratio),
        )
        groups[key].append(cfg)

    results = []
    prepared_cache: Dict[tuple, PreparedData] = {}

    for (emb_path, feature_set, ds_ratio), group_cfgs in tqdm(groups.items(), desc='Experiment groups'):
        print(f"\n{'='*60}")
        cache_key = (emb_path, feature_set, ds_ratio)
        if cache_key not in prepared_cache:
            print(f"Preparing data: feature_set={feature_set}, downsample={ds_ratio}")
            prepared_cache[cache_key] = prepare_evaluation_data(
                df_features=df_features,
                embedding_location_path=emb_path,
                feature_set=feature_set,
                downsample_ratio=ds_ratio,
            )
        prepared_data = prepared_cache[cache_key]

        for cfg in group_cfgs:
            model = cfg['ml_model_object']
            exp_name = cfg['exp_name']
            apply_scaling = cfg.get('apply_scaling', False)
            print(f"  Evaluating: {exp_name} ({type(model).__name__})")
            result = evaluate_with_prepared_data(
                prepared_data=prepared_data,
                ml_model_object=model,
                exp_name=exp_name,
                apply_scaling=apply_scaling,
            )
            results.append(result)

    return pd.DataFrame(results)

## 7. SHAP Feature Importance Function

In [ ]:
# =============================================================================
# SHAP FEATURE IMPORTANCE (model-agnostic)
# Goal: quantify what proportion of top-N features are embeddings vs tabular.
# =============================================================================

def compute_shap_feature_importance(
    fitted_model,
    X_eval: pd.DataFrame,
    feature_cols: List[str],
    embedding_features: List[str],
    top_k_list: List[int] = [10, 20, 50],
    max_samples: int = 2000,
    random_state: int = 42,
    model_name: str = "",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute SHAP feature importance and embedding-vs-tabular breakdown.

    Returns:
        shap_summary_df : [feature, mean_abs_shap, rank, is_embedding]
        proportion_df   : [model_name, top_k, n_embedding_in_top_k,
                           proportion_embedding, n_tabular_in_top_k, proportion_tabular]
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"SHAP: {model_name or type(fitted_model).__name__}")
        print(f"{'='*60}")

    X_sample = X_eval
    if len(X_eval) > max_samples:
        X_sample = X_eval.sample(n=max_samples, random_state=random_state)
        if verbose:
            print(f"  Sampled {max_samples} of {len(X_eval)} rows")

    model_type = type(fitted_model).__name__
    if model_type == 'CatBoostClassifier':
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_sample)
    elif model_type in ('XGBClassifier', 'LGBMClassifier'):
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_sample)
    elif model_type == 'LogisticRegression':
        background = shap.sample(X_sample, min(100, len(X_sample)))
        explainer = shap.LinearExplainer(fitted_model, background)
        shap_values = explainer.shap_values(X_sample)
    else:
        background = shap.sample(X_sample, min(100, len(X_sample)))
        explainer = shap.KernelExplainer(fitted_model.predict_proba, background)
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    embedding_set = set(embedding_features)

    shap_summary_df = pd.DataFrame({
        'feature':       feature_cols,
        'mean_abs_shap': mean_abs_shap,
        'is_embedding':  [f in embedding_set for f in feature_cols],
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    shap_summary_df['rank'] = range(1, len(shap_summary_df) + 1)

    proportion_rows = []
    for k in top_k_list:
        k_actual = min(k, len(shap_summary_df))
        top_k_df = shap_summary_df.head(k_actual)
        n_emb = int(top_k_df['is_embedding'].sum())
        n_tab = k_actual - n_emb
        proportion_rows.append({
            'model_name':           model_name,
            'top_k':                k_actual,
            'n_embedding_in_top_k': n_emb,
            'proportion_embedding': n_emb / k_actual,
            'n_tabular_in_top_k':   n_tab,
            'proportion_tabular':   n_tab / k_actual,
        })

    proportion_df = pd.DataFrame(proportion_rows)
    if verbose:
        print("\nEmbedding proportion in top-K features:")
        print(proportion_df.to_string(index=False))
        print("\nTop 20 features by mean |SHAP|:")
        print(shap_summary_df.head(20)[['rank', 'feature', 'mean_abs_shap', 'is_embedding']].to_string(index=False))

    return shap_summary_df, proportion_df

## 8. Model Configurations

In [ ]:
# Standard CatBoost model
catboost_model = CatBoostClassifier(
    iterations=2500,
    depth=7,
    learning_rate=0.025,
    grow_policy='SymmetricTree',
    auto_class_weights='Balanced',
    od_wait=80,
    use_best_model=True,
    random_seed=42,
    verbose=0,
)

# Legacy-matched configuration (replicates best baseline hyperparameters)
catboost_model_legacy = CatBoostClassifier(
    iterations=2436,
    depth=7,
    learning_rate=0.027,          # rounded from 0.026766501358942353
    random_strength=3,
    l2_leaf_reg=2.95,
    border_count=136,
    min_data_in_leaf=30,
    grow_policy='SymmetricTree',
    od_wait=84,
    bootstrap_type='Bernoulli',
    subsample=0.79,
    leaf_estimation_iterations=8,
    loss_function='Logloss',
    eval_metric='AUC',
    od_type='Iter',
    use_best_model=True,
    random_seed=42,
    thread_count=-1,
    verbose=0,
)

## 9. Load Tabular Feature Table

In [ ]:
feature_sql = f"SELECT * FROM `{FEATURES_TABLE}`"
print(f"Loading: {FEATURES_TABLE}")
df_ip_features = client.query(feature_sql).to_dataframe()
print(f"Loaded {len(df_ip_features):,} rows, {len(df_ip_features.columns)} columns")

In [ ]:
# Quick EDA
print("Target distribution:")
print(df_ip_features[TARGET_COLUMN].value_counts())
print(f"\nPrevalence: {df_ip_features[TARGET_COLUMN].mean()*100:.2f}%")

df_ip_features['index_dt'] = pd.to_datetime(df_ip_features['index_dt']).dt.strftime('%Y-%m-%d')

print("\nIndex date range:")
print(f"  Min: {df_ip_features['index_dt'].min()}")
print(f"  Max: {df_ip_features['index_dt'].max()}")

print("\nind_id_last_digit distribution:")
print(df_ip_features['ind_id_last_digit'].value_counts().sort_index())

## 10. Experiment Configurations

In [ ]:
# All three feature sets share the same pre-computed embedding table.
# Data preparation is cached — each unique (embedding_path, feature_set) pair
# is prepared exactly once.

experiment_configs = [
    # ==================================================================
    # NEW TE EMBEDDINGS (exp_round10_exp2b)
    # ==================================================================
    # ------------------------------------------------------------------
    # Embedding-only: pure transformer signal
    # ------------------------------------------------------------------
    {
        'embedding_location_path': EMBEDDING_TABLE,
        'ml_model_object': catboost_model,
        'exp_name': 'exp_round10_exp2b_catboost_embedding_only',
        'feature_set': 'embedding_only',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Tabular-only: replicates baseline CatBoost pipeline
    # Scoped to members that have embeddings for fair comparison.
    # ------------------------------------------------------------------
    {
        'embedding_location_path': EMBEDDING_TABLE,
        'ml_model_object': catboost_model,
        'exp_name': 'exp_round10_exp2b_catboost_tabular_only',
        'feature_set': 'tabular_only',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Hybrid: tabular + embedding (expected best)
    # ------------------------------------------------------------------
    {
        'embedding_location_path': EMBEDDING_TABLE,
        'ml_model_object': catboost_model,
        'exp_name': 'exp_round10_exp2b_catboost_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Tabular-only (legacy hyperparams) — baseline comparison
    # ------------------------------------------------------------------
    {
        'embedding_location_path': EMBEDDING_TABLE,
        'ml_model_object': catboost_model_legacy,
        'exp_name': 'exp_round10_exp2b_catboost_legacy_tabular_only',
        'feature_set': 'tabular_only',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Hybrid (legacy hyperparams)
    # ------------------------------------------------------------------
    {
        'embedding_location_path': EMBEDDING_TABLE,
        'ml_model_object': catboost_model_legacy,
        'exp_name': 'exp_round10_exp2b_catboost_legacy_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
    # ==================================================================
    # PRODUCTION RAP EMBEDDINGS
    # ==================================================================
    # ------------------------------------------------------------------
    # Prod embedding-only: pure production transformer signal
    # ------------------------------------------------------------------
    {
        'embedding_location_path': PROD_EMBEDDING_TABLE,
        'ml_model_object': catboost_model,
        'exp_name': 'prod_rap_catboost_embedding_only',
        'feature_set': 'embedding_only',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Prod tabular-only: scoped to members in prod embedding table
    # ------------------------------------------------------------------
    {
        'embedding_location_path': PROD_EMBEDDING_TABLE,
        'ml_model_object': catboost_model,
        'exp_name': 'prod_rap_catboost_tabular_only',
        'feature_set': 'tabular_only',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Prod hybrid: tabular + production embedding
    # ------------------------------------------------------------------
    {
        'embedding_location_path': PROD_EMBEDDING_TABLE,
        'ml_model_object': catboost_model,
        'exp_name': 'prod_rap_catboost_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
    # ------------------------------------------------------------------
    # Prod hybrid (legacy hyperparams)
    # ------------------------------------------------------------------
    {
        'embedding_location_path': PROD_EMBEDDING_TABLE,
        'ml_model_object': catboost_model_legacy,
        'exp_name': 'prod_rap_catboost_legacy_hybrid',
        'feature_set': 'hybrid',
        'apply_scaling': False,
    },
]

## 11. Run Evaluation

In [ ]:
results_df = evaluate_all_experiments(
    experiment_configs=experiment_configs,
    df_features=df_ip_features,
    downsample_ratio=10.0,   # match baseline 10:1 negative sampling
)

## 12. Results

In [ ]:
# Full results transposed for readability
results_df.T

In [ ]:
# Key metrics summary — sorted by test AUC-ROC
key_cols = [
    'exp_name', 'feature_set', 'n_features',
    'test_auc_roc', 'test_lift_1pct', 'test_lift_5pct', 'test_lift_10pct',
    'test_precision_1pct', 'test_n_samples', 'test_n_positives', 'test_prevalence',
    'oot_auc_roc', 'oot_lift_1pct', 'oot_lift_5pct', 'oot_lift_10pct',
    'oot_precision_1pct', 'oot_n_samples', 'oot_n_positives', 'oot_prevalence',
]
available_cols = [c for c in key_cols if c in results_df.columns]
results_df[available_cols].sort_values('test_auc_roc', ascending=False)

In [ ]:
# OOT-strict results (digit-9 members only in post-cutoff period)
oot_strict_cols = [
    'exp_name', 'feature_set',
    'oot_strict_auc_roc', 'oot_strict_lift_1pct', 'oot_strict_lift_5pct',
    'oot_strict_precision_1pct', 'oot_strict_n_samples', 'oot_strict_n_positives',
]
available_oot_strict = [c for c in oot_strict_cols if c in results_df.columns]
if available_oot_strict:
    results_df[available_oot_strict].sort_values('oot_strict_auc_roc', ascending=False)

## 13. SHAP Feature Importance — New TE Hybrid Model

In [ ]:
# Train a dedicated hybrid CatBoost for SHAP analysis
prepared_hybrid = prepare_evaluation_data(
    df_features=df_ip_features,
    embedding_location_path=EMBEDDING_TABLE,
    feature_set='hybrid',
    downsample_ratio=10.0,
)

catboost_shap = clone(catboost_model)
cat_idx = prepared_hybrid.cat_feature_indices or []
train_pool_shap = Pool(
    prepared_hybrid.X_splits['train'],
    prepared_hybrid.y_splits['train'],
    cat_features=cat_idx,
)
val_pool_shap = Pool(
    prepared_hybrid.X_splits['val'],
    prepared_hybrid.y_splits['val'],
    cat_features=cat_idx,
)
catboost_shap.fit(train_pool_shap, eval_set=val_pool_shap, verbose=0)
print("Hybrid CatBoost trained for SHAP analysis.")

In [ ]:
commercial_shap_df, commercial_proportion_df = compute_shap_feature_importance(
    fitted_model=catboost_shap,
    X_eval=prepared_hybrid.X_splits['test'],
    feature_cols=prepared_hybrid.feature_cols,
    embedding_features=prepared_hybrid.embedding_features,
    top_k_list=[10, 20, 50],
    max_samples=5000,
    model_name='commercial_round10_exp2b_catboost_hybrid',
    verbose=True,
)

In [ ]:
# Embedding-vs-tabular proportion breakdown
commercial_proportion_df

In [ ]:
# Top features by mean |SHAP|
commercial_shap_df.head(30)

## 13b. SHAP Feature Importance — Production RAP Hybrid Model

In [ ]:
# Train a dedicated hybrid CatBoost using production RAP embeddings for SHAP analysis
prepared_prod_hybrid = prepare_evaluation_data(
    df_features=df_ip_features,
    embedding_location_path=PROD_EMBEDDING_TABLE,
    feature_set='hybrid',
    downsample_ratio=10.0,
)

catboost_shap_prod = clone(catboost_model)
cat_idx_prod = prepared_prod_hybrid.cat_feature_indices or []
train_pool_shap_prod = Pool(
    prepared_prod_hybrid.X_splits['train'],
    prepared_prod_hybrid.y_splits['train'],
    cat_features=cat_idx_prod,
)
val_pool_shap_prod = Pool(
    prepared_prod_hybrid.X_splits['val'],
    prepared_prod_hybrid.y_splits['val'],
    cat_features=cat_idx_prod,
)
catboost_shap_prod.fit(train_pool_shap_prod, eval_set=val_pool_shap_prod, verbose=0)
print("Production RAP Hybrid CatBoost trained for SHAP analysis.")

In [ ]:
prod_shap_df, prod_proportion_df = compute_shap_feature_importance(
    fitted_model=catboost_shap_prod,
    X_eval=prepared_prod_hybrid.X_splits['test'],
    feature_cols=prepared_prod_hybrid.feature_cols,
    embedding_features=prepared_prod_hybrid.embedding_features,
    top_k_list=[10, 20, 50],
    max_samples=5000,
    model_name='prod_rap_catboost_hybrid',
    verbose=True,
)

In [ ]:
# Side-by-side embedding proportion comparison: New TE vs Production RAP
comparison_proportion = pd.concat([commercial_proportion_df, prod_proportion_df], ignore_index=True)
comparison_proportion

In [ ]:
# Top features by mean |SHAP| — Production RAP hybrid
prod_shap_df.head(30)

## 14. Save Results to Excel

In [ ]:
os.makedirs('experiment_logs', exist_ok=True)

output_path = (
    'experiment_logs/'
    'commercial_ip_formal_training_20241120_20250930_downstream_eval.xlsx'
)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Overall metrics
    results_df.T.to_excel(writer, sheet_name='all_metrics')

    # New TE SHAP
    commercial_shap_df.to_excel(writer, sheet_name='new_te_shap_summary', index=False)
    commercial_proportion_df.to_excel(writer, sheet_name='new_te_shap_proportion', index=False)

    # Production RAP SHAP
    prod_shap_df.to_excel(writer, sheet_name='prod_rap_shap_summary', index=False)
    prod_proportion_df.to_excel(writer, sheet_name='prod_rap_shap_proportion', index=False)

    # Combined embedding proportion comparison
    comparison_proportion.to_excel(writer, sheet_name='shap_proportion_comparison', index=False)

print(f"Saved → {output_path}")